# [KNB] BN Griessbreiviertel

In [5]:
import geoppi
from geoppi import (transfer_LoadPoint_ppi, extract_FluidProperties_ppi, get_dict_from_aggregated_groups, implement_controllers, )
import rasterio
import geopandas as gp
import pandas as pd
import numpy as np
from pathlib import Path
import pandapipes as ppi
import os

np.seterr(divide='ignore')

### Define filepaths and load input data
_knb_gis_dir = Path(r"Z:\23_KNB\99_GIS\11_Griesbreiviertel_BN")
cs = "EPSG:25832"

_flp_out = _knb_gis_dir / Path(r"_results_network")
os.makedirs(_flp_out, exist_ok = True)

save_data = True

# Load data
buildings = gp.read_file(_knb_gis_dir / Path(r"Wärmebedarfsmodell/wbm_knb_BN_2026-10-14.gpkg")).to_crs(cs)
buildings = buildings[buildings["in_KNB"].eq(True)].reset_index(drop = True)

lines = gp.read_file(_knb_gis_dir / Path(r"Wärmeleitungen/Netz_Grießbrei_Entwurf_1_2026-10-21.gpkg")).to_crs(cs)
producers =gp.read_file(_knb_gis_dir / Path(r"Erzeugerstandorte/Erzeugerstandorte_polygone_Netzauslegung.gpkg")).to_crs(cs)
dgm = rasterio.open(_knb_gis_dir / Path(r"ALKIS_Grundlagenlayer_WBM/dgm_cut.tif"))


In [6]:
# *--- Parameters for network creation ---*

# Define unique ID for identification of buildings
buildings_uniqueID = 'build_ID'

if buildings_uniqueID not in buildings.columns:
    buildings[buildings_uniqueID] = np.arange(len(buildings))

# Define attribute containing heating demand information
heatingDemandAttr = 'demand_use_th'

# * --- Parameters for network pipe dimensioning ---*

### Network type (either "KMR" or "5GDHC")
networkType = '5GDHC'

### Set temperatures in the network (°C)
tFeed = 5
tReflux = 1

# Set network pressure at peak load producers (bar)
pNetwork = 6

### Valves
# Selection of valves to be closed ("-1": all closed)
valves_closed = [-1]

# Full load hours for calculation of heating power from consumers' heating demand
flh = 1800
thermalPowerAttr = "Pth_kW_source"
buildings[thermalPowerAttr] = buildings[heatingDemandAttr] / flh / 2.5 * (2.5-1)

# Define attribute name containing infotmation about distribution ('VE') and house connection pipes ('HA')
connectionTypeAttr = 'connectionType'

# Define naming convention for base load producers in heat_consumer component DataFrame
baseLoadProdNaming = 'prod'

# Target specific pressure loss in pipes (Pa/m, usually 60-100 Pa/m)
dp_spec = 100

# Selection of subset for nominal widths for pipes
subsetDistribution = [63, 75, 90, 110, 125, 140, 160, 180, 200]
subsetConnection = [32, 40, 50, 63, 75, 90, 110]

# Desired insulation for pipes
insulationType = None

# Medium in the network
fluid = 'water'

# Factor for elongation of pipes to consider influence of armatures etc. on pressure loss
factor_l = 1.3

# Pressure loss calculation model for pandapipes
friction_model = 'swamee-jain'

# Selection if real heights at junctions shall be considered
respect_height = False

# Access network type-dependent data for pipes
dictNominalInnerDiameter, dictWallRoughness = geoppi.load_hydraulic_pipe_characteristics(networkType)

In [7]:
netFeed = geoppi.create_ppi_network_from_gdf(
    pipes = lines,
    buildings = buildings,
    valves = None,
    producers = producers,
    rasterHeight = dgm,
    buildings_uniqueID = buildings_uniqueID,
    heatingDemandAttr = heatingDemandAttr,
    networkName = 'knb_griessbreiviertel_geoppi',
    networkFluid = fluid,
    producerTypeDict = {'Type':{'peak':'peak','base':'base'}},
    producerAddAttr = ['Pth_kW', 'dp_bar_internal'],
    modelling_type = 'feed_line'
)


### Pandapipes network (feed line only) is generated from input GeoDataFrames of network components. ###

... Attention! Global precision of line coordinates is limited to 3 decimal digits!

### The set of start- and end-junctions of all pipes coincides with the total sum of created junctions. Continuing... ###

######### Assignment of junctions to pipes DONE #########

### Function "match_polygons_to_points_by_intersection": 8 of 9 polygons are found with matching points. ###

### Attention! Buildings with following unique IDs could not be matched with junction:  [5520]  ###

### Pandapipes network pipes receive attribute "connectionType" to discriminate between "distribution" and "connection" pipes. ###

### The following columns were found in the provided gdf junction and are NOT REPLACED with default values:
['height_m'] ###


### The following columns were found in the provided gdf pipe and are NOT REPLACED with default values:
['ID', 'from_junction', 'to_junction'] ###


### The

c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pandapower\create.py:5049: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  net[table] = pd.concat([net[table], dd[dd.columns[~dd.isnull().all()]]], sort=False)
c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pandapipes\create.py:1131: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  net.junction_geodata = pd.concat([net.junction_geodata, pd.DataFrame(



### Attention! Attribute Layer not found in junctions columns.
 Skipping this attribute. ###

Modelling of pipes


c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pandapower\create.py:5049: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  net[table] = pd.concat([net[table], dd[dd.columns[~dd.isnull().all()]]], sort=False)
c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pandapipes\create.py:1830: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  net[geo_table] = pd.concat([net[geo_table],df], sort=False)



### Attention! Attribute Layer not found in pipes columns.
 Skipping this attribute. ###


### Attention! Attribute subgraph not found in pipes columns.
 Skipping this attribute. ###

Modelling of sinks


c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pandapower\create.py:5049: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  net[table] = pd.concat([net[table], dd[dd.columns[~dd.isnull().all()]]], sort=False)



### Attention! Attribute Layer not found in sinks columns.
 Skipping this attribute. ###


### Attention! Attribute geometry not found in sinks columns.
 Skipping this attribute. ###


### Pandapipes network for one-directional flow (feed line only) is created.
 Junctions, pipes, valves are created.
 Consumers/buildings are modelled as mass flow sinks.
 Production sites are modelled as external grids (peak load producer) or as mass flow sources. ###


### Peak load producers are modelled as external grids.
               Base load producers are modelled as mass flow sources. ###

### Function "match_polygons_to_points_by_intersection": 2 of 2 polygons are found with matching points. ###

### Attention! Attribute dp_bar_internal not found in columns.
                                 Skipping this attribute while creating producer with index Index(['Type', 'Pth_kW', 'geometry', 'flow_junction'], dtype='object'). ###


### Attention! Attribute dp_bar_internal not found in columns.
      

In [8]:
netFeed = transfer_LoadPoint_ppi(
    net = netFeed,
    tFeed = tFeed + 273.15,
    tReflux = tReflux + 273.15,
    pNetwork = pNetwork,
    heatingDemandAttr = heatingDemandAttr,
    thermalPowerAttrConsumers = thermalPowerAttr,
    baseLoadProdNaming = baseLoadProdNaming,
    check_for_mass_flow_exceeding = True
)

# %% Start algorithm for iterative network dimensioning
cp_fluid, rho_fluid, nu_fluid, g = extract_FluidProperties_ppi(net = netFeed, t = 0.5 * (tFeed + tReflux))

# Adapt user defined options of the network to hydraulic and thermal calculation mode
ppi.set_user_pf_options(net = netFeed, mode="hydraulics", friction_model = friction_model, quit_on_inconsistency_connectivity=True, reset = True)

netFeed = geoppi.hydraulicDimensioningNetwork_singleLoadPoint(
    net = netFeed,
    networkType = networkType,
    dpSpec = dp_spec,
    dictNominalInnerDiameter = {'distributionPipes':dictNominalInnerDiameter, 'connectionPipes':dictNominalInnerDiameter},
    dictWallRoughness = dictWallRoughness,
    subsetDistribution = subsetDistribution,
    subsetConnection = subsetConnection,
    connectionTypeAttr = connectionTypeAttr,
    elongationFactorPipes = factor_l,
    nominalWidthAttr = 'nominalWidth',
    closedValvesIdx = valves_closed,
    frictionModel = friction_model,
    respectHeight = False,
    fluidProperties = {'rho':rho_fluid, 'cp':cp_fluid, 'nu':nu_fluid},
    greaterWidthFinalStep={'distributionPipes':0, 'connectionPipes':0}
)

### Assign final heat transfer coefficient
netFeed.pipe = geoppi.assign_insulation_type(
    df_pipes = netFeed.pipe, 
    colNameNominalWidth = 'nominalWidth', 
    network_type = networkType, 
    insulation_type = insulationType if networkType == 'KMR' else None
)

netFeed = geoppi.update_ppi_results(
    net = netFeed,
    user_pf_options={'mode':'hydraulics', 'friction_model':'swamee-jain'},
    respectHeight = False,
    fluidProperties = {'rho':rho_fluid, 'cp':cp_fluid, 'nu':nu_fluid},
    elongationFactorPipes = factor_l,
    resultAttributes={ 'pipe': ['Pa_per_m', 'mdot_from_kg_per_s'],'junction': ['t_k', 'p_bar'] }
)

if save_data:
    ppi.to_pickle(netFeed, filename = str(_flp_out / Path(r"netFeed.p")))
    gp.GeoDataFrame(netFeed.pipe, geometry = "geometry").to_file(_flp_out / Path(r"pipes_netFeed.gpkg"))
    # gp.GeoDataFrame(netFeed.pipe, geometry = "geometry").to_file(_flp_out / Path(r"pipes_netFeed.gpkg"))


### Total heating demand of consumers connected to network: 1388.0 MWh ###

### Total thermal power of consumers connected to network at load point: 462.7 kW ###
n### Radial system is present and its dimensioning is desired. ###


c:\Users\Voelzel\miniconda3\envs\env_geoppi\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [ ]:
# %% *--- Optional: Intermediate plots ---*
import matplotlib.pyplot as plt

gdf = gp.GeoDataFrame(netFeed.pipe, geometry = 'geometry').set_crs(cs)
gdf['mdot_kg_per_s'] = abs(gdf['mdot_from_kg_per_s'])

fig, ax1 = plt.subplots(3, figsize = (30,20))

styles = {'linewidth':5}
xfont = {'fontsize': 20, 'weight':'normal'}  # Font for axes
tfont = {'fontsize': 20}  # Font for title

ax1[0].set_title('Nominal widths', **tfont)
gdf[(gdf['in_service'] ==True)].plot(
    ax = ax1[0], 
    column = 'nominalWidth', 
    cmap=plt.get_cmap('Paired'),
    categorical = True,
    missing_kwds = dict(color = 'grey', label = '-'),
    legend = True,
    legend_kwds = {'title':'nominal widths.', 
                   'loc':'lower right',
                   'markerscale':1, 
                    'title_fontsize':'xx-large', 
                    'fontsize':'xx-large'},
    **styles)
leg = ax1[0].get_legend()
leg.set_bbox_to_anchor((1.1,0,0.2,0.5))


if not all(gdf.loc[(gdf['in_service'] ==True) & (gdf['connectionType'] == 'distribution'), 'loop'].isna()):
    ax1[1].set_title('Affiliation with loop', **tfont)
    gdf[(gdf['in_service'] ==True) & (gdf[connectionTypeAttr] == 'distribution')].plot(
        ax = ax1[1], 
        column = 'loop', 
        cmap=plt.get_cmap('Paired'),
        legend = True,
        categorical = True,
        missing_kwds = dict(color = 'grey', label = '-'),
        legend_kwds = {'title':'Loop no.', 
                    'loc':'lower right',
                    'markerscale':1, 
                        'title_fontsize':'xx-large', 
                        'fontsize':'xx-large'},
        **styles)
    leg = ax1[1].get_legend()
    leg.set_bbox_to_anchor((1.1,0,0.2,0.5))

ax1[2].set_title('Spec. pressure loss', **tfont)
gdf[(gdf['in_service'] ==True) & (gdf[connectionTypeAttr] == 'distribution')].plot(
    ax = ax1[2], 
    column = 'Pa_per_m', 
    cmap=plt.get_cmap('magma'),
    vmin = 0,
    vmax = 200, 
    legend = True,
    **styles)